# 1. Objectif

Construire un mini-GPT pédagogique, exécutable, et suffisamment fidèle à un GPT moderne pour rendre les concepts transférables.

**Parcours comprendre** : exécuter les chapitres 1 à 18, avec tokenizer caractère et attention manuelle.

**Parcours passer à l'échelle** : essayer ensuite les options indépendantes du chapitre 19 (corpus externe, tailles, BPE, RoPE, cache et SDPA).

# 2. Vue d'ensemble d'un GPT

```
Texte
↓
Tokenizer
↓
Token IDs
↓
Embedding
↓
Position
↓
Self Attention
↓
Multi-Head Attention
↓
Feed Forward
↓
Transformer Blocks
↓
Logits
↓
Next Token
```

In [ ]:
from pathlib import Path
import importlib.util
import torch
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'mini_gpt':
    BASE_DIR = BASE_DIR / 'mini_gpt'

def load_local_module(filename: str, module_name: str):
    path = BASE_DIR / filename
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# 3. Tokenisation

**Concept** : le texte doit devenir une suite d'entiers.

**Intuition** : le réseau ne lit pas des chaînes de caractères, il lit des indices.

**Formule simple** :

- texte → tokens
- tokens → ids

In [ ]:
tok_mod = load_local_module('01_tokenizer.py', 'tok_mod')
text = (BASE_DIR / 'data' / 'tiny_corpus.txt').read_text(encoding='utf-8')
tokenizer = tok_mod.SimpleTokenizer.from_text(text)
sample = 'bonjour'
ids = tokenizer.encode(sample)
print('texte =', sample)
print('tokens =', list(sample))
print('ids =', ids)
print('decode(ids) =', tokenizer.decode(ids))
print('vocab_size =', tokenizer.vocab_size)

# 4. Embeddings

**Concept** : chaque id entier devient un vecteur dense.

**INPUT** : `ids` de shape `[batch_size, sequence_length]`

**OUTPUT** : embeddings de shape `[batch_size, sequence_length, embedding_dim]`

In [ ]:
ids_tensor = torch.tensor([ids], dtype=torch.long)
embedding = torch.nn.Embedding(tokenizer.vocab_size, 8)
embedded = embedding(ids_tensor)
print('ids_tensor shape =', list(ids_tensor.shape))
print('embedded shape =', list(embedded.shape))
print(embedded)

# 5. Positional Encoding / Position Embedding

Sans information de position, les tokens `abc` et `cba` contiendraient les mêmes caractères mais pas le même ordre.

Ici nous utilisons des **position embeddings appris**.

In [ ]:
positions = torch.arange(ids_tensor.size(1)).unsqueeze(0)
pos_embedding = torch.nn.Embedding(ids_tensor.size(1), 8)
pos = pos_embedding(positions)
combined = embedded + pos
print('positions shape =', list(positions.shape))
print('pos shape =', list(pos.shape))
print('combined shape =', list(combined.shape))

# 6. Q / K / V

**Query** = ce que cherche le token

**Key** = ce que propose le token

**Value** = l'information transmise si le token est sélectionné

In [ ]:
att_mod = load_local_module('03_attention.py', 'att_mod')
X = torch.randn(1, 4, 8)
Wq = torch.randn(8, 8)
Wk = torch.randn(8, 8)
Wv = torch.randn(8, 8)
Q = X @ Wq
K = X @ Wk
V = X @ Wv
print('X shape =', list(X.shape))
print('Q shape =', list(Q.shape))
print('K shape =', list(K.shape))
print('V shape =', list(V.shape))

# 7. Attention

Formule:

$$Attention(Q, K, V) = softmax(frac{QK^T}{\sqrt{d_k}})V$$

Pourquoi diviser par $\sqrt{d_k}$ ? Pour éviter que les scores deviennent trop grands, ce qui rendrait le softmax trop extrême.

In [ ]:
out, weights, scaled_scores = att_mod.scaled_dot_product_attention(Q, K, V)
print('QK^T / sqrt(d_k) shape =', list(scaled_scores.shape))
print('weights shape =', list(weights.shape))
print('output shape =', list(out.shape))
print('weights =
', weights)

# 8. Causal Mask

GPT est **decoder-only** et ne doit pas regarder le futur.

Position 1 → voit 1

Position 2 → voit 1,2

Position 3 → voit 1,2,3

Position 4 → voit 1,2,3,4

In [ ]:
mask = att_mod.causal_mask(4)
print(mask.int())
plt.figure(figsize=(4, 4))
plt.imshow(mask.int(), cmap='gray_r')
plt.title('Causal mask')
plt.colorbar()
plt.show()

# 9. Multi-head Attention

Plusieurs têtes peuvent apprendre des relations différentes entre les tokens.

In [ ]:
mha = att_mod.MultiHeadSelfAttention(embedding_dim=8, num_heads=4)
output, attn = mha(torch.randn(1, 4, 8), return_attention=True)
print('attention heads shape =', list(attn.shape))
print('output shape =', list(output.shape))

# 10. Transformer Block

Flux complet :

`x -> self attention -> résidu + layernorm -> feed forward -> résidu + layernorm`

In [ ]:
tr_mod = load_local_module('04_transformer.py', 'tr_mod')
block = tr_mod.TransformerBlock(embedding_dim=128, num_heads=4, ffn_dim=512)
y = block(torch.randn(2, 16, 128), verbose=True)
print('block output shape =', list(y.shape))

# 11. Mini-GPT

On empile plusieurs Transformer Blocks puis on projette vers le vocabulaire pour obtenir des **logits**.

In [ ]:
model_mod = load_local_module('05_model.py', 'model_mod')
model = model_mod.MiniGPT(vocab_size=tokenizer.vocab_size, context_length=16, embedding_dim=64, num_heads=4, num_layers=2, ffn_dim=256, dropout=0.1)
print('nombre total de paramètres =', model.count_parameters())

# 12. Forward Pass

Nous vérifions les shapes à toutes les étapes.

In [ ]:
batch = torch.randint(0, tokenizer.vocab_size, (2, 16))
logits, _ = model(batch, verbose=True)
print('logits final shape =', list(logits.shape))

# 13. Loss

Cross Entropy mesure l'écart entre la distribution prédite et le vrai prochain token.

Shapes avant loss :

- logits : `[B, T, vocab_size]`
- targets : `[B, T]`

Shapes pour `CrossEntropyLoss` :

- logits aplatis : `[B*T, vocab_size]`
- targets aplatis : `[B*T]`

In [ ]:
targets = torch.randint(0, tokenizer.vocab_size, (2, 16))
criterion = torch.nn.CrossEntropyLoss()
loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
print('loss =', float(loss))

# 14. Backpropagation

`loss.backward()` calcule les gradients, c'est-à-dire la direction locale de mise à jour des poids pour réduire la loss.

In [ ]:
model.zero_grad(set_to_none=True)
loss.backward()
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, 'grad mean abs =', float(param.grad.abs().mean()))
        break

# 15. Training

On entraîne le modèle sur `tiny_corpus.txt` avec `AdamW`.

In [ ]:
train_mod = load_local_module('06_train.py', 'train_mod')
checkpoint = train_mod.train()
print(checkpoint['config'])

# 16. Génération

On compare plusieurs températures.

- `temperature = 0.2` : plus déterministe
- `temperature = 0.7` : compromis
- `temperature = 1.2` : plus créatif mais plus risqué

In [ ]:
gen_mod = load_local_module('07_generate.py', 'gen_mod')
model_loaded, tok_loaded = gen_mod.load_model_and_tokenizer()
for temp in [0.2, 0.7, 1.2]:
    print('\nTEMP =', temp)
    text = gen_mod.generate(model_loaded, tok_loaded, prompt='bonjour ', max_new_tokens=10, temperature=temp, top_k=5, sample=True)
    print('texte généré =', text)

# 17. Analyse

Le mode par défaut privilégie la compréhension : tokenizer caractère, corpus minuscule, contexte court et peu de paramètres. Le CPU suffit, mais devient lent lorsque l'échelle augmente.

Le chapitre 19 propose BPE, RoPE, KV cache et attention optimisée comme options. L'entraînement distribué reste hors périmètre. Un modèle plus gros ne produit pas automatiquement de meilleurs textes : il faut aussi assez de données de qualité.

# 18. Du mini-GPT aux vrais LLM

Différences majeures :

- **tokenizer** : BPE / SentencePiece
- **position** : souvent RoPE plutôt que de simples embeddings appris
- **efficacité** : KV cache et Flash Attention
- **échelle** : milliards de paramètres, énormes corpus, entraînement distribué, mixed precision
- **post-entraînement** : fine-tuning, instruction tuning, préférence / RLHF
- **systèmes** : RAG, tool calling, agents, orchestrations de type LangGraph

## Exercices

1. Passe `embedding_dim` de 64 à 128.
2. Passe `num_heads` de 4 à 8. Quelle contrainte faut-il respecter ?
3. Désactive temporairement le causal mask. Que peut apprendre le modèle ?
4. Affiche les poids d'attention et compare 1 tête vs 4 têtes.

### Corrections

1. Le modèle devient plus large et contient plus de paramètres.
2. Il faut garder `embedding_dim % num_heads == 0`.
3. Le modèle fuit l'information du futur pendant l'entraînement.
4. Plusieurs têtes peuvent se spécialiser sur des relations différentes.

# 19. Passer à l'échelle

Les défauts restent inchangés. Presets : `tiny` (contexte 32, dimension 64, 4 têtes, 2 couches), `small` (128, 128, 4, 4), `medium` (256, 256, 8, 6). Chaque dimension est configurable ; RoPE exige une dimension par tête paire.

Le corpus UTF-8 est lu par blocs, séparé en train/validation avant toute tokenisation ; seul le train apprend le vocabulaire. Les IDs sont stockés sur disque et lus par memory mapping. Les frontières de blocs bornent aussi les fusions BPE. Pour borner les statistiques BPE, son apprentissage utilise au plus le premier million de caractères du train, par blocs de 4096 caractères ; choisir un début de corpus représentatif. Tout le train est ensuite encodé pour le modèle. Le vocabulaire byte-level comporte au moins 257 entrées (256 octets et <unk>). Les fenêtres train sont échantillonnées avec remise pour éviter une permutation d'indices géante.

BPE est facultatif : installer `requirements-bpe.txt` depuis le dossier absolu du projet avant de choisir `tokenizer_kind='bpe'`. Le corpus fourni est trop petit pour des expériences BPE ou de grands contextes. Les deux portions doivent dépasser le contexte en nombre de tokens.

In [ ]:
# Démonstration courte, CPU, caractère : RoPE et SDPA sont indépendants de BPE.
import tempfile

with tempfile.TemporaryDirectory(prefix='mini-gpt-notebook-') as output_dir:
    advanced_checkpoint = train_mod.train(
        checkpoint_dir=Path(output_dir),
        preset='tiny',
        tokenizer_kind='char',
        position_encoding='rope',
        attention_backend='sdpa',
        device='cpu',
        max_steps=2,
    )
    advanced_model, advanced_tokenizer = gen_mod.load_model_and_tokenizer(
        Path(output_dir) / 'mini_gpt.pt', device='cpu'
    )
print(advanced_checkpoint['metrics'])

## Corpus externe et BPE (expérience facultative)

Dans `train_mod.train`, remplacer `corpus_path` par le chemin absolu d'un grand corpus local, choisir `preset='small'`, `tokenizer_kind='bpe'`, `vocab_size=2000` et un répertoire de checkpoint dédié. Ajuster `stride`, `batch_size`, `epochs`, `learning_rate` et `max_steps` au budget. Sur GPU compatible, utiliser `device='cuda'` et `precision='bfloat16'`.

Le tokenizer complet est sauvegardé dans le checkpoint. Les anciens checkpoints caractère restent chargeables. RoPE exige un nouvel entraînement ; changer le backend manuel/SDPA ne change pas les poids.

SDPA sélectionne Flash Attention seulement lorsque matériel, dtype et formes le permettent ; sinon PyTorch choisit un autre backend, dont un repli CPU. Ce n'est pas une garantie de Flash. L'attention manuelle reste disponible pour visualiser les poids.

In [ ]:
# Inclut le dépassement du contexte : cache reconstruit sur la fenêtre glissante.
without_cache = gen_mod.generate(
    advanced_model, advanced_tokenizer, 'bonjour ',
    max_new_tokens=40, temperature=0, use_cache=False, verbose=False,
)
with_cache = gen_mod.generate(
    advanced_model, advanced_tokenizer, 'bonjour ',
    max_new_tokens=40, temperature=0, use_cache=True, verbose=False,
)
print('Textes gloutons identiques :', without_cache == with_cache)
print(with_cache)

## Comparer honnêtement

Le cache ne sert qu'en inférence : il réutilise les K/V jusqu'à remplir le contexte. Au débordement, la fenêtre entière est recalculée avec positions remises à zéro, ce qui préserve le comportement sans cache mais réduit le gain de vitesse.

Le CLI de génération propose `--benchmark` (échauffement, avec/sans cache, temps, tokens/s, pic mémoire CUDA et comparaison des textes). Comparer manuel/SDPA sur un même checkpoint, prompt, dtype et matériel ; répéter les mesures. De faibles différences numériques peuvent modifier la génération.

L'entraînement rapporte les losses pondérées par token, le débit validation comprise, le nombre de paramètres et leur taille. La mémoire des paramètres n'est pas la mémoire totale ; le pic CPU n'est pas mesuré. Ne pas comparer directement les losses ni les tokens/s caractère et BPE : les tokens représentent des quantités de texte différentes. Examiner aussi la qualité sur des prompts identiques, le temps pour une quantité comparable de texte et la mémoire.

Les tests de non-régression sont exécutables avec `python -m unittest discover -s /home/runner/work/SHOKOBO/SHOKOBO/mini_gpt/tests -v`.